In [11]:
# ============================================================
# TRANSFORM CIRCUITS DATA — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql import functions as F
from pyspark.sql.types import *
import nbformat
from nbconvert import PythonExporter

In [12]:
# -----------------------------------------
# 1. Load environment + helpers
# -----------------------------------------
def run_notebook(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    source, _ = PythonExporter().from_notebook_node(nb)
    exec(source, globals())

run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/03_silver_helpers.ipynb")

===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [13]:
# -----------------------------------------
# 2. Receive batch_id from pipeline (unified logic)
# -----------------------------------------

# 1. Papermill parameter (local pipeline)
if "p_batch_id" in globals() and p_batch_id not in [None, ""]:
    pass

# 2. Databricks widget (only if running in Databricks)
elif "dbutils" in globals():
    p_batch_id = dbutils.widgets.get("p_batch_id")

# 3. Manual execution fallback (Jupyter)
else:
    # AUTO-DETECT latest batch_id from Bronze
    bronze_batches = sorted([
        f.split("=")[1]
        for f in os.listdir(f"{BRONZE_PATH}/circuits/data.parquet")
        if f.startswith("batch_id=")
    ])
    if not bronze_batches:
        raise Exception("❌ No batch_id found in Bronze folder")
    p_batch_id = bronze_batches[-1]
    print("⚠️ Auto-selected latest batch_id:", p_batch_id)

# Final validation
if p_batch_id is None or p_batch_id == "":
    raise Exception("❌ p_batch_id not provided to Silver notebook")

print("Using p_batch_id:", p_batch_id)


⚠️ Auto-selected latest batch_id: 20260609_130719
Using p_batch_id: 20260609_130719


In [14]:
# -----------------------------------------
# 3. Define Bronze + Silver paths
# -----------------------------------------
bronze_path = f"{BRONZE_PATH}/circuits/data.parquet"
silver_path = f"{SILVER_PATH}/circuits"
os.makedirs(silver_path, exist_ok=True)

In [15]:
# -----------------------------------------
# 4. Read Bronze (ONLY this batch)
# -----------------------------------------
circuits_df = (
    spark.read.parquet(bronze_path)
    .filter(F.col("batch_id") == p_batch_id)
)

print("✔ Bronze circuits read")
circuits_df.show(5)

✔ Bronze circuits read
+---------+--------------------+--------------------+--------+--------+----------+---------+--------------------+--------------------+---------------+
|circuitId|                 url|         circuitName|     lat|    long|  locality|  country|    ingest_timestamp|         source_file|       batch_id|
+---------+--------------------+--------------------+--------+--------+----------+---------+--------------------+--------------------+---------------+
|     NULL|https://en.wikipe...|circuit gilles vi...|    45.5|-73.5228|  montreal|   Canada|2026-06-09 13:07:...|/Users/manoharaza...|20260609_130719|
|     NULL|https://en.wikipe...|losail internatio...|   25.49| 51.4542|    lusail|    Qatar|2026-06-09 13:07:...|/Users/manoharaza...|20260609_130719|
| adelaide|https://en.wikipe...|adelaide street c...|-34.9272| 138.617|  adelaide|Australia|2026-06-09 13:07:...|/Users/manoharaza...|20260609_130719|
| ain-diab|https://en.wikipe...|            ain diab| 33.5786| -7.6875|

In [16]:
# -----------------------------------------
# 5. Select required columns
# -----------------------------------------
circuits_selected_df = circuits_df.select(
    "circuitId",
    "circuitName",
    "lat",
    "long",
    "locality",
    "country",
    "ingest_timestamp",
    "source_file",
    "batch_id"
)

In [17]:
# -----------------------------------------
# 6. Standardize + rename columns
# -----------------------------------------
circuits_renamed_df = (
    circuits_selected_df
        .withColumnRenamed("circuitId", "circuit_id")
        .withColumnRenamed("circuitName", "circuit_name")
        .withColumnRenamed("lat", "latitude")
        .withColumnRenamed("long", "longitude")
)

In [18]:
# -----------------------------------------
# 7. Business key validation
# -----------------------------------------
circuits_valid_df = circuits_renamed_df.filter("circuit_id IS NOT NULL")

In [19]:
# -----------------------------------------
# 8. Remove duplicates
# -----------------------------------------
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

In [20]:
# -----------------------------------------
# 9. Title Case transformations
# -----------------------------------------
circuits_final_df = (
    circuits_distinct_df
        .withColumn("circuit_name", F.initcap("circuit_name"))
        .withColumn("locality", F.initcap("locality"))
        .withColumnRenamed("country", "country_name")
)

In [21]:
# -----------------------------------------
# 10. Write to Silver
# -----------------------------------------
write_to_silver(
    circuits_final_df,
    f"{silver_path}/data.parquet",
    merge_keys=["circuit_id"]
)

print("✔ Circuits Silver written:", f"{silver_path}/data.parquet")

✔ Circuits Silver written: /Users/manoharazalki/F1-Analytics/data/silver/circuits/data.parquet


In [22]:
# -----------------------------------------
# 11. Validate
# -----------------------------------------
spark.read.parquet(f"{silver_path}/data.parquet").show(5)

+-----------+--------------------+--------+---------+----------+------------+--------------------+--------------------+---------------+--------------------+--------------------+
| circuit_id|        circuit_name|latitude|longitude|  locality|country_name|    ingest_timestamp|         source_file|       batch_id|   created_timestamp|   updated_timestamp|
+-----------+--------------------+--------+---------+----------+------------+--------------------+--------------------+---------------+--------------------+--------------------+
|   adelaide|Adelaide Street C...|-34.9272|  138.617|  Adelaide|   Australia|2026-06-09 13:07:...|/Users/manoharaza...|20260609_130719|2026-06-09 13:36:...|2026-06-09 13:36:...|
|   ain-diab|            Ain Diab| 33.5786|  -7.6875|Casablanca|     Morocco|2026-06-09 13:07:...|/Users/manoharaza...|20260609_130719|2026-06-09 13:36:...|2026-06-09 13:36:...|
|    aintree|             Aintree| 53.4769| -2.94056| Liverpool|          UK|2026-06-09 13:07:...|/Users/manoh